# Mobile Money Agent Network: Audit and Clean-Up

**Analyst:** Marion Oduor · PLP Data Analytics 2026 Cohort

The regional manager wants to know which agents and counties handle the most deposits and withdrawals, but the extract from the legacy system is unreliable. This notebook audits the data, cleans it with the reusable function `clean_agent_data()` from `cleaning_pipeline.py`, merges it with the agent register, produces the summary tables and ends with a Data Quality Report for the data governance team.

## Setup and loading the raw data

If the files are missing (for example in a fresh Colab session), the setup cell downloads the repository files and runs `generate_data.py`, which recreates `agents.csv` and `agent_transactions.csv` with the fixed seed 44. The transactions are loaded twice on purpose: once normally, and once as exact text with `keep_default_na=False`, so that the audit can tell a blank cell from the text "N/A" (pandas converts both to missing by default).

In [1]:
import os
import subprocess
import sys
import urllib.request

import numpy as np
import pandas as pd

REPO_RAW = ("https://raw.githubusercontent.com/Reeescyberspacexx/"
            "plp-da-a3-duka-sales-and-agent-cleanup/main/")


def ensure_file(path):
    """Download a repository file if it is not already available locally."""
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + path, path)


ensure_file("generate_data.py")
ensure_file("cleaning_pipeline.py")
if not (os.path.exists("agent_transactions.csv") and os.path.exists("agents.csv")):
    subprocess.run([sys.executable, "generate_data.py"], check=True)
os.makedirs("outputs", exist_ok=True)

from cleaning_pipeline import clean_agent_data


def show(df, decimals=0):
    """Display a DataFrame with thousands separators on decimal columns."""
    with pd.option_context("display.float_format", f"{{:,.{decimals}f}}".format):
        display(df)

In [2]:
txns_raw = pd.read_csv("agent_transactions.csv")     # pandas default: "N/A" becomes NaN
agents = pd.read_csv("agents.csv")
raw_text = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)  # exact text

print("Transactions:", txns_raw.shape, "| Agent register:", agents.shape)
display(txns_raw.head(8))
display(agents.head())
txns_raw.info()
print("\nMissing amounts when pandas loads the file normally:", txns_raw["amount"].isna().sum())
print("Blank amounts in the raw text:", (raw_text["amount"].str.strip() == "").sum())
print("'N/A' amounts in the raw text:", (raw_text["amount"].str.strip().str.upper() == "N/A").sum())

Transactions: (612, 5) | Agent register: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17
5,TX00006,AG019,withdrawal,931.0,2026-04-23
6,TX00007,AG020,deposit,2583.0,2026-04-02
7,TX00008,AG019,deposit,1797.0,2026-04-12


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


<class 'pandas.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    str    
 1   agent_id  612 non-null    str    
 2   txn_type  612 non-null    str    
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    str    
dtypes: float64(1), str(4)
memory usage: 24.0 KB

Missing amounts when pandas loads the file normally: 22
Blank amounts in the raw text: 15
'N/A' amounts in the raw text: 7


**First look:** the extract has 612 transaction rows and the register lists 20 agents. Loading normally, pandas reports 22 missing amounts, but the raw text shows these are really 15 blank cells plus 7 entries of the text "N/A". Because pandas hides that difference, the audit below works on the raw text as well.

## Task 1: Data-Quality Audit

**Approach:** Check every column against the six data-quality dimensions and record each problem found as one row of an audit table: the issue, the dimension it violates, the column affected and the number of rows affected. All counts are calculated in code from the raw extract.

In [3]:
known_agents = set(agents["agent_id"])
standard_types = {"Deposit", "Withdrawal"}
amount_numeric = pd.to_numeric(raw_text["amount"], errors="coerce")
extreme_value = amount_numeric.max()
unknown_ids = sorted(set(raw_text["agent_id"]) - known_agents)

audit = pd.DataFrame(
    [
        ("Duplicate transactions (the same txn_id appears more than once)",
         "Uniqueness", "txn_id", int(raw_text["txn_id"].duplicated().sum())),
        ("Transaction type spelled inconsistently (case, trailing spaces)",
         "Consistency", "txn_type", int((~raw_text["txn_type"].isin(standard_types)).sum())),
        ("Amount missing (blank cell)",
         "Completeness", "amount", int((raw_text["amount"].str.strip() == "").sum())),
        ("Amount recorded as the text 'N/A'",
         "Validity", "amount", int((raw_text["amount"].str.strip().str.upper() == "N/A").sum())),
        (f"Identical extreme amount of KES {extreme_value:,.0f} (suspected error)",
         "Accuracy", "amount", int((amount_numeric == extreme_value).sum())),
        ("Dates written with '/' instead of '-'",
         "Consistency", "txn_date", int(raw_text["txn_date"].str.contains("/").sum())),
        (f"Agent ID not in the agent register ({', '.join(unknown_ids)})",
         "Consistency", "agent_id", int((~raw_text["agent_id"].isin(known_agents)).sum())),
    ],
    columns=["issue", "dimension", "column", "rows_affected"],
)
pd.set_option("display.max_colwidth", None)
display(audit)
print("Rows in the raw extract:", len(raw_text))

,issue,dimension,column,rows_affected
0,Duplicate transactions (the same txn_id appears more than once),Uniqueness,txn_id,12
1,"Transaction type spelled inconsistently (case, trailing spaces)",Consistency,txn_type,384
2,Amount missing (blank cell),Completeness,amount,15
3,Amount recorded as the text 'N/A',Validity,amount,7
4,"Identical extreme amount of KES 250,000 (suspected error)",Accuracy,amount,5
5,Dates written with '/' instead of '-',Consistency,txn_date,5
6,Agent ID not in the agent register (AG099),Consistency,agent_id,8


Rows in the raw extract: 612


**Reading the audit:** the counts come from the raw 612-row extract, so a duplicated row is counted wherever it appears. Seven issues were found across five of the six dimensions. The sixth, **timeliness**, was checked and no problem was found (see Task 4). Two details deserve attention:

- Of the 12 duplicated `txn_id` values, 11 are exact copies of another row, but 1 differs in `agent_id` (one copy names the unregistered agent AG099 and the other a registered agent). Removing only exact duplicates would have left a duplicate `txn_id` behind.
- The 5 amounts of KES 250,000 are identical to the shilling, which is unusual for real transactions (investigated in Task 6).

## Running the cleaning pipeline

**Approach:** All cleaning rules live in one function, `clean_agent_data(txns, agents)`, in `cleaning_pipeline.py`. It returns the cleaned DataFrame and a log of what it changed. The sections for Tasks 2 to 7 below call the function once and then check each rule against the result.

In [4]:
clean, cleaning_log = clean_agent_data(txns_raw, agents)

for line in cleaning_log:
    print("-", line)
print("\nClean data:", clean.shape)

- Started with 612 transaction rows and 20 registered agents
- Uniqueness: removed 12 duplicate txn_id rows (11 exact copies, plus 1 that differed in another column, where the copy with a registered agent was kept); 600 rows remain
- Consistency: standardised 376 txn_type values to Deposit / Withdrawal; 0 could not be mapped
- Validity: 20 amounts are blank, 'N/A' or not a valid positive number (0 zero or negative)
- Validity: converted txn_date to datetime (5 dates used '/' separators); 0 dates could not be parsed
- Outliers: IQR fences are 0 to 6,084; flagged 40 amounts in is_outlier (none deleted)
- Outliers: the identical value KES 250,000 appears 5 times among the flagged amounts, which is suspicious
- Completeness: filled 20 missing amounts with the median of the real amounts (KES 1,776) and flagged them in amount_imputed
- Merge: left join kept 600 of 600 rows; 7 transactions reference an agent that is not in the register and were given the county 'Unknown'

Clean data: (600, 8)

## Task 2: Uniqueness (Duplicate Transactions)

**Approach:** Duplicates are defined by `txn_id`, the transaction's unique key, rather than by whole-row equality. When copies of the same transaction differ, the pipeline keeps the copy whose agent exists in the register.

In [5]:
print("Rows before:", len(txns_raw), "| rows after:", len(clean), "| removed:", len(txns_raw) - len(clean))
print("Duplicate txn_id values remaining:", clean["txn_id"].duplicated().sum())
print("Exact duplicate rows in the raw data:", txns_raw.duplicated().sum())

repeated = txns_raw[txns_raw.duplicated("txn_id", keep=False)]
differing = repeated[~repeated.duplicated(keep=False)].sort_values("txn_id")
print("\nRaw rows for a txn_id whose copies are not identical:")
display(differing)
print("What the clean data kept for it:")
display(clean[clean["txn_id"].isin(differing["txn_id"])])

Rows before: 612 | rows after: 600 | removed: 12
Duplicate txn_id values remaining: 0
Exact duplicate rows in the raw data: 11

Raw rows for a txn_id whose copies are not identical:


,txn_id,agent_id,txn_type,amount,txn_date
404,TX00405,AG014,Deposit,1936.0,2026-04-03
601,TX00405,AG099,Deposit,1936.0,2026-04-03


What the clean data kept for it:


,txn_id,agent_id,county,txn_type,amount,txn_date,amount_imputed,is_outlier
404,TX00405,AG014,Nairobi,Deposit,1936.0,2026-04-03,False,False


**Answer:** **12 duplicate transactions were removed**, taking the data from 612 to 600 rows. 11 were exact copies. The other 1 (`TX00405`) had two copies that disagreed on the agent; the pipeline kept the copy with the registered agent, so no valid information was lost. No `txn_id` is repeated in the clean data.

## Task 3: Consistency (Transaction Type)

**Approach:** Strip spaces, convert to lower case and map to exactly two labels, "Deposit" and "Withdrawal". Unmapped values would become missing and be counted in the log.

In [6]:
print("Before (repr shows trailing spaces):")
print(txns_raw["txn_type"].map(repr).value_counts().to_string())
print("\nAfter:")
print(clean["txn_type"].value_counts().to_string())
print("\nOnly two values remain:", sorted(clean["txn_type"].unique()))
print("Missing txn_type values:", clean["txn_type"].isna().sum())

Before (repr shows trailing spaces):
txn_type
'deposit'       142
'withdrawal'    131
'Deposit'       128
'DEPOSIT '      111
'Withdrawal'    100

After:
txn_type
Deposit       373
Withdrawal    227

Only two values remain: ['Deposit', 'Withdrawal']
Missing txn_type values: 0


**Answer:** the five spellings (`Deposit`, `deposit`, `DEPOSIT ` with a trailing space, `Withdrawal`, `withdrawal`) are now exactly two: Deposit (373 transactions) and Withdrawal (227). Without this step, grouping by type would have produced five groups and understated both totals.

## Task 4: Validity and Data Types

**Approach:** Convert `amount` with `pd.to_numeric(..., errors="coerce")` after stripping spaces and commas, so blanks, "N/A" and any other text become missing instead of crashing. Convert `txn_date` to datetime after replacing "/" with "-", using an explicit `%Y-%m-%d` format. The pipeline also treats zero or negative amounts as invalid, and the log reports how many there were.

In [7]:
print(clean.dtypes, "\n")

slash_ids = raw_text.loc[raw_text["txn_date"].str.contains("/"), "txn_id"]
print("Rows whose date used '/' in the raw data, now parsed as:")
display(clean[clean["txn_id"].isin(slash_ids)][["txn_id", "txn_date"]])

print("Unparseable dates:", clean["txn_date"].isna().sum())
print("Missing or non-positive amounts after cleaning:", (clean["amount"].isna() | (clean["amount"] <= 0)).sum())
show(clean["amount"].describe().to_frame())

txn_id                    string
agent_id                  string
county                       str
txn_type                     str
amount                   float64
txn_date          datetime64[us]
amount_imputed              bool
is_outlier                  bool
dtype: object 

Rows whose date used '/' in the raw data, now parsed as:


,txn_id,txn_date
87,TX00088,2026-04-27
128,TX00129,2026-04-28
181,TX00182,2026-03-20
437,TX00438,2026-03-30
571,TX00572,2026-03-04


Unparseable dates: 0
Missing or non-positive amounts after cleaning: 0


,amount
count,600
mean,"4,481"
std,"22,648"
min,109
25%,"1,105"
50%,"1,776"
75%,"2,868"
max,"250,000"


In [8]:
# Timeliness check (used in the Data Quality Report): period covered, and any transaction
# dated before the agent was onboarded.
agents["onboarded"] = pd.to_datetime(agents["onboarded"])
timeliness = clean.merge(agents[["agent_id", "onboarded"]], on="agent_id", how="left")
print(f"Period covered: {clean['txn_date'].min():%d %b %Y} to {clean['txn_date'].max():%d %b %Y}")
print("Transactions dated before the agent's onboarding date:",
      (timeliness["txn_date"] < timeliness["onboarded"]).sum())

Period covered: 01 Mar 2026 to 29 Apr 2026
Transactions dated before the agent's onboarding date: 0


**Answer:** every `amount` is now numeric and every `txn_date` is a real datetime, with no unparseable dates. The 5 dates written with "/" were converted along with the rest. The raw `amount` column contained text that is not a number ("N/A") and blank cells, which were turned into missing values for the next step. The transactions run from 01 March 2026 to 29 April 2026 (60 days), and 0 transactions are dated before their agent's onboarding date.

## Task 5: Completeness (Missing and Invalid Amounts)

**Approach:** Flag first, then fill. Every missing or invalid amount is marked `amount_imputed = True`, and only then replaced with the median of the real amounts, rounded to whole shillings.

In [9]:
imputed = clean[clean["amount_imputed"]]
real = clean.loc[~clean["amount_imputed"], "amount"]

print("Rows with an imputed amount:", len(imputed), f"({len(imputed) / len(clean) * 100:.1f}% of rows)")
print(f"Median of real amounts (used to fill): KES {real.median():,.0f}")
print(f"Mean of real amounts (not used):       KES {real.mean():,.0f}")
print(f"Value added by imputation: KES {imputed['amount'].sum():,.0f} "
      f"({imputed['amount'].sum() / clean['amount'].sum() * 100:.1f}% of total value)")
display(imputed.head())

Rows with an imputed amount: 20 (3.3% of rows)
Median of real amounts (used to fill): KES 1,776
Mean of real amounts (not used):       KES 4,574
Value added by imputation: KES 35,520 (1.3% of total value)


,txn_id,agent_id,county,txn_type,amount,txn_date,amount_imputed,is_outlier
10,TX00011,AG009,Nakuru,Withdrawal,1776.0,2026-03-27,True,False
14,TX00015,AG003,Kisumu,Deposit,1776.0,2026-03-18,True,False
54,TX00055,AG012,Kisumu,Withdrawal,1776.0,2026-03-19,True,False
66,TX00067,AG002,Nairobi,Deposit,1776.0,2026-04-07,True,False
122,TX00123,AG008,Kisumu,Deposit,1776.0,2026-04-15,True,False


**Decision and justification:** the 20 missing or invalid amounts (15 blank and 5 "N/A" after duplicates were removed) were **flagged in `amount_imputed` and filled with the median, KES 1,776**.

- **Why not delete the rows?** Each one is a real deposit or withdrawal that happened; only the amount was lost. Dropping them would remove 3.3% of transactions and understate activity for the agents and counties involved.
- **Why the median and not the mean?** The amounts are right-skewed and contain the extreme values examined in Task 6. The mean of the real amounts is KES 4,574, more than twice the median, so filling with the mean would invent large transactions that never happened.
- **Why flag first?** Once filled, an imputed value looks like a real one. The flag lets the business re-run any total without the imputed rows, and the imputed rows account for only KES 35,520 (1.3% of total value).
- **Limitation:** every imputed row gets the same value, so individual imputed amounts are guesses. The proper fix is for the legacy system to stop exporting blank or "N/A" amounts.

## Task 6: Outliers (IQR Method)

**Approach:** Calculate Q1, Q3 and the interquartile range (IQR) from the real, non-imputed amounts. Amounts above Q3 + 1.5 × IQR are flagged in `is_outlier`. Flagged rows are investigated, not deleted automatically.

In [10]:
real = clean.loc[~clean["amount_imputed"], "amount"]
q1, q3 = real.quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 = {q1:,.0f} | Q3 = {q3:,.0f} | IQR = {iqr:,.0f}")
print(f"Fences: {max(lower_fence, 0):,.0f} to {upper_fence:,.0f}")
print("Rows flagged as outliers:", clean["is_outlier"].sum())

print("\nThe ten largest amounts:")
display(clean.nlargest(10, "amount")[["txn_id", "agent_id", "county", "txn_type", "amount", "txn_date"]])

Q1 = 1,077 | Q3 = 3,080 | IQR = 2,003
Fences: 0 to 6,084
Rows flagged as outliers: 40

The ten largest amounts:


,txn_id,agent_id,county,txn_type,amount,txn_date
306,TX00307,AG012,Kisumu,Deposit,250000.0,2026-03-02
315,TX00316,AG014,Nairobi,Deposit,250000.0,2026-04-01
424,TX00425,AG005,Mombasa,Withdrawal,250000.0,2026-03-02
463,TX00464,AG009,Nakuru,Withdrawal,250000.0,2026-04-11
594,TX00595,AG016,Kiambu,Deposit,250000.0,2026-03-11
183,TX00184,AG006,Nakuru,Deposit,28847.0,2026-03-08
222,TX00223,AG011,Kisumu,Deposit,22316.0,2026-03-02
573,TX00574,AG017,Nakuru,Deposit,16605.0,2026-03-16
174,TX00175,AG014,Nairobi,Withdrawal,13362.0,2026-04-06
212,TX00213,AG019,Mombasa,Deposit,12078.0,2026-04-10


In [11]:
top_value = clean["amount"].max()
suspects = clean[clean["amount"] == top_value]
others = clean[clean["is_outlier"] & (clean["amount"] != top_value)]
rest = clean.loc[clean["amount"] != top_value, "amount"]

print(f"Amounts equal to KES {top_value:,.0f}: {len(suspects)}")
display(suspects[["txn_id", "agent_id", "county", "txn_type", "txn_date"]])
print(f"Other flagged amounts: {len(others)}, from KES {others['amount'].min():,.0f} to KES {others['amount'].max():,.0f}")
print(f"Largest amount excluding the suspects: KES {rest.max():,.0f} "
      f"({top_value / rest.max():.1f} times smaller than KES {top_value:,.0f})")
print(f"Share of the other amounts that are exact multiples of 1,000: {(real[real != top_value] % 1000 == 0).mean() * 100:.1f}%")
print(f"The {len(suspects)} suspects add up to KES {suspects['amount'].sum():,.0f}, "
      f"{suspects['amount'].sum() / clean['amount'].sum() * 100:.1f}% of all recorded value")

Amounts equal to KES 250,000: 5


,txn_id,agent_id,county,txn_type,txn_date
306,TX00307,AG012,Kisumu,Deposit,2026-03-02
315,TX00316,AG014,Nairobi,Deposit,2026-04-01
424,TX00425,AG005,Mombasa,Withdrawal,2026-03-02
463,TX00464,AG009,Nakuru,Withdrawal,2026-04-11
594,TX00595,AG016,Kiambu,Deposit,2026-03-11


Other flagged amounts: 35, from KES 6,158 to KES 28,847
Largest amount excluding the suspects: KES 28,847 (8.7 times smaller than KES 250,000)
Share of the other amounts that are exact multiples of 1,000: 0.0%
The 5 suspects add up to KES 1,250,000, 46.5% of all recorded value


**Investigation:** the IQR rule flags **40 amounts** above KES 6,084, but they are two very different groups.

- **35 ordinary high values** (KES 6,158 to KES 28,847) are what a right-skewed set of amounts produces naturally. They are plausible large deposits or withdrawals and should be **kept**.
- **5 amounts of exactly KES 250,000** are different. They are 8.7 times larger than the next-largest amount (KES 28,847), identical to the shilling, and the share of the other amounts that are exact multiples of 1,000 is 0.0%. They are spread over 5 counties and both transaction types, and together make up KES 1,250,000, or **46.5% of all recorded value**. A repeated round number looks like a system default, a transaction ceiling or test data rather than 5 independent real transactions.

**Recommendation:** do not delete them automatically, and do not trust them either. (1) Verify the 5 `txn_id` values against the source system or agent receipts, and ask the system owner whether KES 250,000 is a default or limit. (2) Until confirmed, report totals both with and without them (Task 8 does this) and keep them out of averages. (3) Keep them in the data with `is_outlier = True` so the decision can be reversed.

## Task 7: Merge with the Agent Register and Post-Merge Validation

**Approach:** Left-join the transactions to the register on `agent_id` with `validate="many_to_one"` (many transactions per agent, one register row per agent) and `indicator=True`. A left join keeps every transaction, and agents that are not in the register get the county "Unknown" instead of being silently dropped.

In [12]:
rows_after_dedup = txns_raw["txn_id"].nunique()
print("Rows before the merge (after removing duplicates):", rows_after_dedup)
print("Rows after the merge:", len(clean))
print("Row count unchanged:", rows_after_dedup == len(clean))

unknown = clean[clean["county"] == "Unknown"]
print("\nTransactions with an agent that is not in the register:", len(unknown))
print(unknown["agent_id"].value_counts().to_string())
print(f"Value of those transactions: KES {unknown['amount'].sum():,.0f} "
      f"({unknown['amount'].sum() / clean['amount'].sum() * 100:.1f}% of total value)")

# What an inner join would have done
inner = clean.merge(agents[["agent_id"]], on="agent_id", how="inner")
print(f"\nAn inner join would return {len(inner)} rows and silently drop "
      f"{len(clean) - len(inner)} transactions worth KES {clean['amount'].sum() - inner['amount'].sum():,.0f}")

print("\nAgents per county in the register:")
print(agents["county"].value_counts().to_string())
print("\nTransactions per county after the merge:")
print(clean["county"].value_counts().to_string())

Rows before the merge (after removing duplicates): 600
Rows after the merge: 600
Row count unchanged: True

Transactions with an agent that is not in the register: 7
agent_id
AG099    7
Value of those transactions: KES 27,156 (1.0% of total value)

An inner join would return 593 rows and silently drop 7 transactions worth KES 27,156

Agents per county in the register:
county
Nairobi    6
Mombasa    5
Kisumu     4
Nakuru     3
Kiambu     2

Transactions per county after the merge:
county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7


**Answer:** the left join kept all 600 rows, with no rows lost or multiplied. **7 transactions (1.2%) reference an unknown agent (AG099)**, worth KES 27,156 (1.0% of recorded value). I kept them and labelled their county "Unknown" rather than deleting them, because an inner join would have silently removed 7 real transactions. They are reported separately in the summaries. Data governance should find out whether AG099 is a typing error, an agent missing from the register, or an unregistered agent.

## Export: Clean Data

The cleaned data is saved with `index=False` and ISO-formatted dates, and then read back to confirm that it meets the requirements.

In [13]:
clean.to_csv("outputs/agent_transactions_clean.csv", index=False, date_format="%Y-%m-%d")

saved = pd.read_csv("outputs/agent_transactions_clean.csv")
print("Rows:", len(saved), "| columns:", list(saved.columns))
print("Duplicate txn_id values:", saved["txn_id"].duplicated().sum())
print("txn_type values:", sorted(saved["txn_type"].unique()))
print("amount is numeric:", pd.api.types.is_numeric_dtype(saved["amount"]))
print("All dates ISO formatted (YYYY-MM-DD):", saved["txn_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all())
print("Flag columns present:", {"amount_imputed", "is_outlier"} <= set(saved.columns))
print("County column present, no missing values:", saved["county"].notna().all())

Rows: 600 | columns: ['txn_id', 'agent_id', 'county', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier']
Duplicate txn_id values: 0
txn_type values: ['Deposit', 'Withdrawal']
amount is numeric: True
All dates ISO formatted (YYYY-MM-DD): True
Flag columns present: True
County column present, no missing values: True


## Task 8: Summary Tables

**Approach:** Group by county and transaction type with named aggregation for (a), then build the weekly pivot table for (b). Because the suspect KES 250,000 transactions would dominate the totals, a second table shows the county totals with and without them.

In [14]:
county_summary = (clean.groupby(["county", "txn_type"])
                  .agg(total_value=("amount", "sum"),
                       transactions=("txn_id", "count"),
                       avg_value=("amount", "mean"))
                  .reset_index())
print("(a) County summary by transaction type (all cleaned transactions):")
show(county_summary)

(a) County summary by transaction type (all cleaned transactions):


,county,txn_type,total_value,transactions,avg_value
0,Kiambu,Deposit,"334,737",43,"7,785"
1,Kiambu,Withdrawal,"77,050",27,"2,854"
2,Kisumu,Deposit,"436,419",74,"5,898"
3,Kisumu,Withdrawal,"76,374",37,"2,064"
4,Mombasa,Deposit,"148,984",72,"2,069"
5,Mombasa,Withdrawal,"368,592",61,"6,042"
6,Nairobi,Deposit,"543,588",118,"4,607"
7,Nairobi,Withdrawal,"162,034",62,"2,613"
8,Nakuru,Deposit,"184,889",60,"3,081"
9,Nakuru,Withdrawal,"328,594",39,"8,425"


In [15]:
suspect_mask = clean["amount"] == clean["amount"].max()      # the repeated extreme amount (Task 6)
county_view = pd.DataFrame({
    "total_value_all": clean.groupby("county")["amount"].sum(),
    "total_value_excl_suspects": clean[~suspect_mask].groupby("county")["amount"].sum(),
    "transactions": clean.groupby("county")["txn_id"].count(),
})
county_view["share_of_value"] = (county_view["total_value_all"] / county_view["total_value_all"].sum() * 100).map("{:.1f}%".format)
county_view = county_view.sort_values("total_value_all", ascending=False)
print("County totals, with and without the suspect KES 250,000 transactions:")
show(county_view)
print("Totals by transaction type:")
show(clean.groupby("txn_type")["amount"].agg(total_value="sum", transactions="count", avg_value="mean"))

County totals, with and without the suspect KES 250,000 transactions:


,total_value_all,total_value_excl_suspects,transactions,share_of_value
county,,,,
Nairobi,"705,622","455,622",180,26.2%
Mombasa,"517,576","267,576",133,19.3%
Nakuru,"513,483","263,483",99,19.1%
Kisumu,"512,793","262,793",111,19.1%
Kiambu,"411,787","161,787",70,15.3%
Unknown,"27,156","27,156",7,1.0%


Totals by transaction type:


,total_value,transactions,avg_value
txn_type,,,
Deposit,"1,673,896",373,"4,488"
Withdrawal,"1,014,521",227,"4,469"


In [16]:
weekly = clean.assign(week_start=clean["txn_date"] - pd.to_timedelta(clean["txn_date"].dt.dayofweek, unit="D"))
weekly_pivot = pd.pivot_table(weekly, index="week_start", columns="county", values="amount",
                              aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
weekly_pivot.index = [d.strftime("%d %b %Y") if hasattr(d, "strftime") else d for d in weekly_pivot.index]
weekly_pivot.index.name = "week starting (Monday)"
print("(b) Weekly transaction value (KES) by county:")
show(weekly_pivot)

(b) Weekly transaction value (KES) by county:


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week starting (Monday),,,,,,,
23 Feb 2026,0,"1,893","6,759","7,046","2,743",0,"18,441"
02 Mar 2026,"18,317","287,933","285,669","68,967","56,574",0,"717,460"
09 Mar 2026,"267,771","23,186","30,477","52,860","28,730",0,"403,024"
16 Mar 2026,"22,054","28,124","25,144","43,107","39,795","1,639","159,863"
23 Mar 2026,"11,504","27,077","17,050","41,373","18,998",0,"116,002"
30 Mar 2026,"22,609","41,077","38,344","282,931","18,145","8,507","411,613"
06 Apr 2026,"15,030","15,341","47,092","56,786","276,609","9,824","420,682"
13 Apr 2026,"12,265","29,415","26,730","56,432","26,152",0,"150,994"
20 Apr 2026,"24,507","37,439","16,687","65,186","30,862","1,877","176,558"


**What the summaries show**

- **County ranking:** with every cleaned transaction included, the order by total value is Nairobi, Mombasa, Nakuru, Kisumu, Kiambu. Nairobi leads with KES 705,622 from 180 transactions (26.2% of value). The order is exactly the same without the 5 suspect KES 250,000 transactions, because each of the five counties has exactly one of them, but they hide how far ahead the leader is: Nairobi's total is 1.36 times the runner-up's with them and 1.70 times without. The counties ranked second to fourth are within 0.9% of each other with the suspects and 1.8% without, so their relative order should not be over-interpreted.
- **Deposits versus withdrawals:** deposits are 62.2% of transactions (373 of 600) and 62.3% of value, so the network handles about 1.6 deposits for every withdrawal. Average amounts are similar for the two types.
- **Unknown agents:** the "Unknown" row (7 transactions, KES 27,156) is shown separately and is not assigned to any real county.
- **Weekly pattern:** the largest week by value starts on 02 March 2026 (KES 717,460), but 2 of the suspect KES 250,000 transactions fall in that week; without them it is KES 217,460. The first week (starting 23 February) contains only 1 March (8 transactions) and the last week is also partial (27 to 29 April), so they should not be compared with full weeks.

## Task 9: The Reusable Cleaning Pipeline

`clean_agent_data(txns, agents)` is saved in `cleaning_pipeline.py` and imported at the top of this notebook. It has a docstring, applies every rule in code with no hard-coded row numbers and no manual edits, and returns `(clean_df, log)`. The check below confirms that it gives an identical result whether the file is loaded normally or as raw text, and shows its docstring.

In [17]:
clean_from_text, log_from_text = clean_agent_data(raw_text, agents)
print("Identical result from the raw-text load:", clean_from_text.equals(clean))
print("Identical log:", log_from_text == cleaning_log)
print()
print(clean_agent_data.__doc__)

Identical result from the raw-text load: True
Identical log: True

Clean the raw agent transactions and attach each agent's county.

    Rules applied, in order (each one is recorded in the log):
      1. Uniqueness: keep one row per txn_id. If copies differ, keep the copy whose
         agent_id exists in the agent register, so no valid information is lost.
      2. Consistency: standardise txn_type to exactly "Deposit" or "Withdrawal".
      3. Validity: convert amount to numbers (blank, "N/A" and other text become
         missing; zero or negative amounts are treated as invalid) and convert
         txn_date to datetime (dates written with "/" are accepted).
      4. Outliers: flag amounts outside the IQR fences, calculated on the real
         (non-imputed) amounts. Flagged rows are kept, never deleted.
      5. Completeness: flag missing amounts in amount_imputed, then fill them with the
         median of the real amounts (rounded to whole shillings).
      6. Merge: left-join t

## Data Quality Report

**Scope:** 612 raw transaction rows from the legacy system and 20 registered agents, covering 01 March to 29 April 2026. After cleaning there are **600 transactions**, each with a unique `txn_id`. Issues are organised by the six data-quality dimensions.

| Dimension | What was found | What was done |
|---|---|---|
| **Completeness** | 20 of 600 amounts (3.3%) were missing: 15 blank and 5 "N/A". | Flagged in `amount_imputed`, then filled with the median (KES 1,776). These rows add only 1.3% of value. |
| **Validity** | Amounts stored as text, including "N/A". After conversion there were no zero or negative amounts and no unparseable dates. | Converted to numbers and dates; invalid text became missing and was handled under Completeness. |
| **Accuracy** | 5 amounts of exactly KES 250,000, 8.7 times the next-largest amount and 46.5% of all value. They are almost certainly not independent real transactions. | Flagged (`is_outlier`), kept, and reported with and without them. Needs verification against the source system. |
| **Consistency** | 384 of 612 raw rows used a non-standard `txn_type` spelling; 5 dates used "/"; 8 raw rows (7 after de-duplication) used agent AG099, which is not in the register. | Types standardised to Deposit and Withdrawal; dates converted to ISO format; unknown-agent rows kept with county "Unknown" (KES 27,156). |
| **Uniqueness** | 12 duplicate `txn_id` rows: 11 exact copies and 1 copy that differed in `agent_id`. | Removed, keeping the copy with a registered agent (612 to 600 rows). |
| **Timeliness** | Data covers 60 days; no date is in the future and 0 transactions precede their agent's onboarding date. No defect found. | None needed. Confirm the extract date and that later transactions exist. |

**Recommendations for data governance:** (1) verify the 5 KES 250,000 transactions and the unregistered agent AG099; (2) change the legacy export to write amounts as plain numbers (never "N/A" or blank) and dates as YYYY-MM-DD; (3) enforce unique transaction IDs and a fixed list of transaction types at the source; (4) keep running `clean_agent_data()` on every extract so that each rule is applied the same way and logged.

**Privacy (Kenya Data Protection Act, 2019):** this extract contains no customer details, but if it also carried customer phone numbers, then phone numbers, names and national ID numbers would be personal data (direct identifiers), and so would any field that could be linked back to a person, such as a customer's full transaction history. Agent IDs could also count where an agent is an individual trader. The Act requires that only data needed for the purpose is processed, so I would first drop every personal field that the county-level analysis does not need. Where customers must be told apart, I would replace the phone number with a salted hash (pseudonymisation), keep the salt outside the repository, show masked numbers such as 0712****78 in any display, and report only aggregates, hiding any group of fewer than about five people. Raw customer files would never be emailed or committed to a public GitHub repository, access would be limited to those who need it, and the data would be deleted once the purpose is complete. Before any real analysis, the organisation's data-protection policy and its Data Protection Officer should be consulted.